# Bài 4.2 · Feature extraction, linear probe và phân cụm

**Câu hỏi của buổi này:** không train mạng nào, chỉ đổi mỗi ảnh thành một vector bằng một encoder có sẵn. Vector ấy tốt đến đâu: ảnh giống nhau có nằm gần nhau, một bộ phân loại tuyến tính đạt bao nhiêu, cần bao nhiêu nhãn?

**CIFAR-10**: ảnh màu 32 × 32, 10 lớp. Lấy 5 000 ảnh train và 2 000 ảnh test. So ba cách biến ảnh thành vector: **pixel thô**, **ResNet-50** (ImageNet), **CLIP** (ghép ảnh với chú thích chữ).

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập, ảnh mẫu |
| 2. Ba cách biến ảnh thành vector | bảng số chiều; ảnh gần nhất của cùng một ảnh hỏi theo từng cách |
| 3. Linear probe và kNN | bảng và biểu đồ accuracy |
| 4. Ít nhãn | accuracy theo số ảnh có nhãn |
| 5. Phân cụm không dùng nhãn | NMI, ARI, bản đồ PCA, ảnh trong từng cụm |
| 6. Văn bản: pooling bỏ padding | cosine khi có và không có mask |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 6 phút. Bài giảng: [ai-canuc80k · bài 4.2](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/4.2).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time
import torch, torchvision, transformers, datasets
from torchvision import transforms as T
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.decomposition import PCA
transformers.logging.set_verbosity_error(); datasets.disable_progress_bars()

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TRAIN, N_TEST = 5000, 2000
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Ảnh giữ dạng PIL; mỗi encoder tự tiền xử lý theo hợp đồng của nó (bài 4.1).

In [ ]:
with quiet():
    cifar = datasets.load_dataset('uoft-cs/cifar10')

class HFImages:
    def __init__(self, split):
        self.d, self.targets, self.classes = split, list(split['label']), split.features['label'].names
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d[int(i)]
        return r['img'].convert('RGB'), r['label']

train_full, test_full = HFImages(cifar['train']), HFImages(cifar['test'])
rng = np.random.default_rng(0)
tr_idx = rng.permutation(len(train_full))[:N_TRAIN]
te_idx = rng.permutation(len(test_full))[:N_TEST]
classes = train_full.classes
y_tr, y_te = np.array(train_full.targets)[tr_idx], np.array(test_full.targets)[te_idx]
table(pd.DataFrame([('train', len(tr_idx), np.bincount(y_tr).min(), np.bincount(y_tr).max()), ('test', len(te_idx), np.bincount(y_te).min(), np.bincount(y_te).max())],
                   columns=['tập', 'số ảnh', 'lớp ít nhất', 'lớp nhiều nhất']), 'Lấy ngẫu nhiên, cố định seed')
fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
for k, ax in enumerate(axes):
    ax.imshow(train_full[tr_idx[np.flatnonzero(y_tr == k)[0]]][0]); ax.set_title(classes[k], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Ba cách biến ảnh thành vector

- **Pixel thô**: duỗi 32 × 32 × 3 thành 3 072 số.
- **ResNet-50**: bỏ lớp cuối, lấy vector 2 048 chiều sau global average pooling; ảnh phóng lên 224.
- **CLIP ViT-B/32**: embedding ảnh 512 chiều.

Model đặt `eval()` và chạy trong `inference_mode()`: không lưu gradient, không cập nhật BatchNorm.

In [ ]:
def pixels(ds, idx):
    return np.stack([np.asarray(ds[i][0], dtype=np.float32).ravel() / 255 for i in idx])

class Wrap(torch.utils.data.Dataset):
    def __init__(self, ds, idx, tf): self.ds, self.idx, self.tf = ds, idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, k): return self.tf(self.ds[self.idx[k]][0])

def embed(fn, ds, idx, tf, batch=128):
    out = []
    with torch.inference_mode():
        for x in DataLoader(Wrap(ds, idx, tf), batch, num_workers=2):
            out.append(fn(x.to(device)).float().cpu())
    return torch.cat(out).numpy()

w = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
with quiet():
    resnet = torchvision.models.resnet50(weights=w)
    from transformers import CLIPModel, CLIPImageProcessor
    clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
    proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-base-patch32')
resnet.fc = torch.nn.Identity(); resnet = resnet.to(device).eval()
res_tf = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(w.transforms().mean, w.transforms().std)])
clip_tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(), T.Normalize(proc.image_mean, proc.image_std)])
feat = lambda out: out if torch.is_tensor(out) else out.pooler_output          # transformers 5 trả object
clip_fn = lambda x: feat(clip.get_image_features(pixel_values=x))

feats, tim = {}, {}
for name, fn in [('pixel thô', lambda d, i: pixels(d, i)), ('ResNet-50', lambda d, i: embed(resnet, d, i, res_tf)), ('CLIP ViT-B/32', lambda d, i: embed(clip_fn, d, i, clip_tf))]:
    t = time.time(); feats[name] = (fn(train_full, tr_idx), fn(test_full, te_idx)); tim[name] = time.time() - t
table(pd.DataFrame([(n, a.shape[1], f'{a.shape[0]} × {a.shape[1]}', tim[n]) for n, (a, b) in feats.items()], columns=['biểu diễn', 'số chiều', 'ma trận train', 'giây (7 000 ảnh)']),
      'Mỗi ảnh thành một vector', fmt={'giây (7 000 ảnh)': '{:.0f}'})

Vector tốt thì ảnh **giống nhau về nội dung** nằm gần nhau. Lấy ba ảnh test làm câu hỏi, tìm 5 ảnh train gần nhất theo cosine với mỗi cách biểu diễn (viền xanh lá: cùng lớp với ảnh hỏi).

In [ ]:
def unit(a): return a / np.linalg.norm(a, axis=1, keepdims=True)
queries = [int(np.flatnonzero(y_te == k)[0]) for k in [classes.index('cat'), classes.index('ship'), classes.index('horse')]]
fig, axes = plt.subplots(len(queries) * 3, 6, figsize=(11, 2 * 3 * len(queries)))
for qi, q in enumerate(queries):
    for ri, (name, (a, b)) in enumerate(feats.items()):
        row = axes[qi * 3 + ri]
        sim = unit(a) @ unit(b[q:q + 1]).T
        nn_ = np.argsort(-sim[:, 0])[:5]
        row[0].imshow(test_full[te_idx[q]][0]); row[0].set_title('ảnh hỏi' if ri == 0 else '', fontsize=9)
        row[0].set_ylabel(name, fontsize=9); row[0].set_xticks([]); row[0].set_yticks([])
        for c, j in enumerate(nn_, start=1):
            row[c].imshow(train_full[tr_idx[j]][0]); row[c].set_xticks([]); row[c].set_yticks([])
            ok = y_tr[j] == y_te[q]
            for s in row[c].spines.values(): s.set_visible(True); s.set_color(C['green'] if ok else C['clay']); s.set_linewidth(3)
        row[1].set_title(name, fontsize=9, loc='left')
plt.tight_layout(); plt.show()
hits = {n: np.mean([[y_tr[j] == y_te[q] for j in np.argsort(-(unit(a) @ unit(b[q:q + 1]).T)[:, 0])[:5]] for q in range(300)]) for n, (a, b) in feats.items()}
note('Tỉ lệ 5 ảnh gần nhất cùng lớp với ảnh hỏi (300 ảnh test): ' + ', '.join(f'{n} <b>{v:.2f}</b>' for n, v in hits.items()) + '.')

## 3. Linear probe và kNN

Cùng một bộ phân loại tuyến tính cho cả ba biểu diễn; chỉ đặc trưng thay đổi. Chuẩn hóa bằng mean, std **của train**. kNN dùng cosine, $k = 10$.

In [ ]:
rows = []
for name, (a, b) in feats.items():
    sc = StandardScaler().fit(a)
    lr = LogisticRegression(max_iter=2000, C=0.5).fit(sc.transform(a), y_tr)
    acc_lr = (lr.predict(sc.transform(b)) == y_te).mean()
    acc_knn = (KNeighborsClassifier(10).fit(unit(a), y_tr).predict(unit(b)) == y_te).mean()
    rows.append((name, a.shape[1], acc_lr, acc_knn))
probe = pd.DataFrame(rows, columns=['biểu diễn', 'số chiều', 'linear probe', 'kNN (k=10)'])
table(probe, f'Accuracy trên {N_TEST} ảnh test', fmt={'linear probe': '{:.3f}', 'kNN (k=10)': '{:.3f}'}, best='linear probe')
ax = probe.set_index('biểu diễn')[['linear probe', 'kNN (k=10)']].plot.barh(figsize=(9, 3.2), color=[C['blue'], C['sand']], width=.75)
ax.set_xlim(0, 1.08); ax.invert_yaxis(); ax.set_ylabel(''); ax.set_title('Cùng bộ phân loại, khác biểu diễn'); ax.legend(loc='lower right'); bar_labels(ax, '{:.3f}')
plt.tight_layout(); plt.show()
check('CLIP vượt pixel thô', probe['linear probe'].iloc[2] > probe['linear probe'].iloc[0] + 0.3)
note(f"Chỉ đổi biểu diễn, accuracy đi từ <b>{probe['linear probe'].iloc[0]:.3f}</b> lên <b>{probe['linear probe'].iloc[2]:.3f}</b>. Trên pixel thô, hai ảnh cùng lớp nhưng khác vị trí, màu nền đã rất xa nhau; encoder pretrained kéo chúng lại gần.")

## 4. Ít nhãn

Biểu diễn tốt thì cần ít nhãn. Học lại linear probe với 100, 500, 2 000 và 5 000 ảnh train, chấm trên cùng 2 000 ảnh test.

In [ ]:
sizes = [100, 500, 2000, 5000]
curve = []
for name, (a, b) in feats.items():
    for n in sizes:
        sc = StandardScaler().fit(a[:n])
        lr = LogisticRegression(max_iter=2000, C=0.5).fit(sc.transform(a[:n]), y_tr[:n])
        curve.append((name, n, (lr.predict(sc.transform(b)) == y_te).mean()))
curve = pd.DataFrame(curve, columns=['biểu diễn', 'số ảnh train', 'acc'])
fig, ax = plt.subplots(figsize=(8, 4))
for (name, g), c in zip(curve.groupby('biểu diễn', sort=False), [C['gray'], C['sand'], C['blue']]):
    ax.plot(g['số ảnh train'], g['acc'], marker='o', color=c, lw=2, label=name)
    ax.annotate(f"{g['acc'].iloc[0]:.2f}", (g['số ảnh train'].iloc[0], g['acc'].iloc[0]), xytext=(-28, 4), textcoords='offset points', fontsize=9, color=c)
ax.set_xscale('log'); ax.set(xlabel='số ảnh train có nhãn (thang log)', ylabel='accuracy test', title='Biểu diễn tốt cần rất ít nhãn', ylim=(0, 1)); ax.legend()
plt.tight_layout(); plt.show()
piv = curve.pivot(index='số ảnh train', columns='biểu diễn', values='acc')[list(feats)].reset_index()
table(piv, 'Accuracy theo số ảnh có nhãn', fmt={n: '{:.3f}' for n in feats})
note(f"Với 100 ảnh (khoảng 10 ảnh mỗi lớp), CLIP đã đạt <b>{piv['CLIP ViT-B/32'].iloc[0]:.3f}</b>, pixel thô {piv['pixel thô'].iloc[0]:.3f}; "
     f"lên 5 000 ảnh, CLIP chỉ nhích tới {piv['CLIP ViT-B/32'].iloc[-1]:.3f}.")

## 5. Phân cụm không dùng nhãn

k-means 10 cụm trên vector đã chuẩn hóa độ dài. NMI và ARI đo mức cụm khớp lớp thật: 1 là khớp hoàn toàn, quanh 0 là như chia ngẫu nhiên. Nhãn chỉ dùng để chấm.

In [ ]:
rows, labs = [], {}
for name, (a, b) in feats.items():
    labs[name] = KMeans(10, n_init=10, random_state=0).fit_predict(unit(b))
    rows.append((name, normalized_mutual_info_score(y_te, labs[name]), adjusted_rand_score(y_te, labs[name])))
clu = pd.DataFrame(rows, columns=['biểu diễn', 'NMI', 'ARI'])
table(clu, 'k-means 10 cụm trên 2 000 ảnh test', fmt={'NMI': '{:.3f}', 'ARI': '{:.3f}'}, best='NMI')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (name, (a, b)) in zip(axes, feats.items()):
    p = PCA(2, random_state=0).fit_transform(StandardScaler().fit_transform(b))
    sc = ax.scatter(p[:, 0], p[:, 1], c=y_te, cmap='tab10', s=5)
    ax.set_title(f'{name}: PCA 2 chiều, màu = lớp thật'); ax.set_xticks([]); ax.set_yticks([])
fig.legend(sc.legend_elements()[0], classes, loc='center right', fontsize=9)
plt.subplots_adjust(right=0.9); plt.show()

In [ ]:
lab = labs['CLIP ViT-B/32']
fig, axes = plt.subplots(5, 8, figsize=(12, 8))
for r, k in enumerate(range(5)):
    members = np.flatnonzero(lab == k)
    major = np.bincount(y_te[members], minlength=10).argmax(); purity = (y_te[members] == major).mean()
    for c, i in enumerate(members[:8]):
        axes[r, c].imshow(test_full[te_idx[i]][0]); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(f'cụm {k}\n{classes[major]} {purity:.0%}', fontsize=9)
fig.suptitle('Năm cụm k-means trên vector CLIP, mỗi cụm 8 ảnh (nhãn chiếm đa số, tỉ lệ)', y=1.0)
plt.tight_layout(); plt.show()
note(f"CLIP cho NMI <b>{clu['NMI'].iloc[2]:.3f}</b>, pixel thô {clu['NMI'].iloc[0]:.3f}. Cụm chia theo những gì gần nhau trong không gian vector; điều đó không nhất thiết trùng cách chia của đề.")

## 6. Văn bản: pooling phải bỏ qua padding

Encoder văn bản trả một vector cho **mỗi token**; muốn một vector cho cả câu thì lấy trung bình các token. Trong một batch, câu ngắn được đệm thêm token padding; tính trung bình cả padding thì vector của câu ngắn đổi theo câu dài nhất trong batch.

In [ ]:
from transformers import AutoTokenizer, AutoModel
name = 'sentence-transformers/all-MiniLM-L6-v2'
with quiet():
    tok = AutoTokenizer.from_pretrained(name); enc = AutoModel.from_pretrained(name).to(device).eval()

def sentence_vec(texts, use_mask=True):
    b = tok(texts, padding=True, return_tensors='pt').to(device)
    with torch.inference_mode():
        h = enc(**b).last_hidden_state
    m = b['attention_mask'].unsqueeze(-1).float() if use_mask else torch.ones_like(h[..., :1])
    return torch.nn.functional.normalize((h * m).sum(1) / m.sum(1), dim=1).cpu()

short = 'A cat sleeps on the sofa.'
long = 'The quarterly financial report shows that revenue grew in every region except the northern market.'
pool = pd.DataFrame([(('có mask' if um else 'không mask'), float(sentence_vec([short], um)[0] @ sentence_vec([short, long], um)[0])) for um in [True, False]],
                    columns=['cách lấy trung bình', 'cosine(câu ngắn chạy riêng, chạy chung batch)'])
table(pool, f'Câu ngắn {len(tok(short).input_ids)} token, câu dài {len(tok(long).input_ids)} token', fmt={'cosine(câu ngắn chạy riêng, chạy chung batch)': '{:.4f}'})
check('có mask thì vector câu ngắn không đổi theo batch', pool.iloc[0, 1] > 0.9999)

## 7. Tóm tắt

In [ ]:
summary = [('linear probe: pixel / ResNet-50 / CLIP', ' / '.join(f'{v:.3f}' for v in probe['linear probe'])),
           ('5 ảnh gần nhất cùng lớp: pixel / ResNet-50 / CLIP', ' / '.join(f'{v:.2f}' for v in hits.values())),
           ('CLIP với 100 ảnh có nhãn', f"{piv['CLIP ViT-B/32'].iloc[0]:.3f}"),
           ('NMI k-means: pixel / ResNet-50 / CLIP', ' / '.join(f'{v:.3f}' for v in clu['NMI'])),
           ('pooling không mask: cosine câu ngắn', f'{pool.iloc[1, 1]:.4f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thay `openai/clip-vit-base-patch32` bằng `openai/clip-vit-base-patch16`; đổi ảnh hỏi ở mục 2 sang lớp khác; giảm `N_TRAIN` xuống 500 rồi so lại các dòng.